In [94]:
import os
import shutil
from git import Repo
from langchain.text_splitter import Language
from langchain.document_loaders.generic import GenericLoader
from langchain.document_loaders.parsers import LanguageParser
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.vectorstores import Chroma
from langchain.chat_models import ChatOpenAI
from langchain.memory import ConversationSummaryMemory
from langchain.chains import ConversationalRetrievalChain

In [95]:
!mkdir test_repo

A subdirectory or file test_repo already exists.


In [96]:
repo_path = "test_repo"

if os.path.exists(os.path.join(repo_path, ".git")):
    repo = Repo(repo_path)
    repo.remotes.origin.pull()
else:
    if os.path.exists(repo_path):
        shutil.rmtree(repo_path)  # clean invalid dir
    repo = Repo.clone_from(
        "https://github.com/entbappy/End-to-end-Medical-Chatbot-Generative-AI/",
        to_path=repo_path
    )

In [97]:
loader = GenericLoader.from_filesystem(
    repo_url, 
    glob="**/*",
    suffixes=[".py"], 
    parser=LanguageParser(
        language=Language.PYTHON,
        parser_threshold=500  # avoids parsing tiny irrelevant files
    ),
    show_progress=True
)

In [98]:
documents = loader.load()

100%|██████████| 7/7 [00:00<00:00, 1163.61it/s]


In [99]:
 documents_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON, 
    chunk_size=500, chunk_overlap=20)

In [100]:
split_docs = documents_splitter.split_documents(documents)    

In [101]:
from dotenv import load_dotenv
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
assert OPENAI_API_KEY, "Missing OPENAI_API_KEY"

In [102]:
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [103]:
embeddings = OpenAIEmbeddings(disallowed_special=())

In [104]:
vectordb = Chroma.from_documents(
    split_docs, 
    embedding=embeddings,
    persist_directory="./chroma_db")

Failed to send telemetry event client_start: capture() takes 1 positional argument but 3 were given


In [105]:
vectordb.persist()

In [106]:
llm = ChatOpenAI()

In [107]:
memory = ConversationSummaryMemory(
    llm=llm, 
    memory_key="chat_history",
    input_key="question",   # ✅ fix
    output_key="answer",
    return_messages=True,
    max_token_limit=800
)

In [108]:
qa = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=vectordb.as_retriever(
        search_type="mmr", 
        search_kwargs={"k": 8}
    ),
    memory=memory   # <-- missing
)

In [109]:
question = "What is download_hugging_face_model function?"

In [110]:
result = qa({
    "question": question, 
    "chat_history": []
    })
    
print(result["answer"])

The `download_hugging_face_embeddings` function downloads embeddings from the Hugging Face model `sentence-transformers/all-MiniLM-L6-v2`, which returns embeddings of 384 dimensions.


In [111]:
import textwrap

answer = result["answer"]

formatted = textwrap.fill(answer, width=80)  # adjust width as needed
print("\n=== Answer ===\n")
print(formatted)


=== Answer ===

The `download_hugging_face_embeddings` function downloads embeddings from the
Hugging Face model `sentence-transformers/all-MiniLM-L6-v2`, which returns
embeddings of 384 dimensions.


In [112]:
from fastapi import FastAPI
from fastapi.responses import HTMLResponse

app = FastAPI()   # ✅ THIS WAS MISSING

@app.get("/answer", response_class=HTMLResponse)
def get_answer():
    answer = result["answer"]

    html_content = f"""
    <div style="font-family: Arial; line-height: 1.6; max-width: 800px; margin: auto;">
        <h2>Answer</h2>
        <p>{answer}</p>
    </div>
    """

    return html_content

In [113]:
from IPython.display import Markdown, display

display(Markdown(f"**Answer:**\n\n{result['answer']}"))

**Answer:**

The `download_hugging_face_embeddings` function downloads embeddings from the Hugging Face model `sentence-transformers/all-MiniLM-L6-v2`, which returns embeddings of 384 dimensions.